<a href="https://colab.research.google.com/github/bangaru01/YVR-Reactions/blob/main/YVR_R1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pubchempy pandas

In [24]:
import pandas as pd
import pubchempy as pcp
import re
from IPython.display import FileLink

reaction_scale = 0.1  # mmol

reaction_reagents = [
    {"Name": "1-vinylnaphthalene", "Equivalents": 1, "State": "liquid", "Density": 0.97},
    {"Name": "morpholine", "Equivalents": 1, "State": "liquid", "Density": 0.999},
    {"Name": "potassium persulfate", "Equivalents": 2, "State": "solid", "Density": None},
    {"Name": "sodium chloride", "Equivalents": 1.5, "State": "solid", "Density": None}
]

data = []

for r in reaction_reagents:

    compound = pcp.get_compounds(r["Name"], 'name')[0]

    mw = compound.molecular_weight

    cas = "Not Found"
    pattern = r"\d{2,7}-\d{2}-\d"

    for syn in compound.synonyms:
        if re.match(pattern, syn):
            cas = syn
            break

    mmol = reaction_scale * r["Equivalents"]

    mass_g = (mw * mmol) / 1000

    volume_ml = ""

    if r["State"] == "liquid" and r["Density"]:
        volume_ml = mass_g / r["Density"]

    data.append({
        "Compound": r["Name"],
        "CAS Number": cas,
        "State": r["State"],
        "MW (g/mol)": mw,
        "Equivalents": r["Equivalents"],
        "mmol": round(mmol,4),
        "Mass (g)": round(mass_g,6),
        "Volume (mL)": round(volume_ml,6) if volume_ml else ""
    })

df = pd.DataFrame(data)

solvent = pd.DataFrame([{
    "Solvent": "Acetonitrile",
    "CAS Number": "75-05-8",
    "Volume (mL)": 1
}])

file_name = "reaction_setup.xlsx"

with pd.ExcelWriter(file_name) as writer:
    df.to_excel(writer, sheet_name="Reagents", index=False)
    solvent.to_excel(writer, sheet_name="Solvent", index=False)

print("Excel file created!")

display(FileLink(file_name))

Excel file created!


/content/reaction_setup.xlsx

In [25]:
df

,Compound,CAS Number,State,MW (g/mol),Equivalents,mmol,Mass (g),Volume (mL)
0,1-vinylnaphthalene,826-74-4,liquid,154.21,1.0,0.10,0.015421,0.015898
1,morpholine,110-91-8,liquid,87.12,1.0,0.10,0.008712,0.008721
2,potassium persulfate,7727-21-1,solid,270.33,2.0,0.20,0.054066,
3,sodium chloride,7647-14-5,solid,58.44,1.5,0.15,0.008766,


In [26]:
temperature = 60   # reaction temperature in °C

procedure = "A reaction vial was charged with "

parts = []

for i,row in df.iterrows():

    if row["Volume (mL)"] != "":
        part = f'{row["Compound"]} ({row["mmol"]} mmol, {row["Volume (mL)"]} mL)'
    else:
        part = f'{row["Compound"]} ({row["mmol"]} mmol, {row["Mass (g)"]} g)'

    parts.append(part)

procedure += ", ".join(parts)

procedure += f". The mixture was dissolved in acetonitrile (1 mL) and stirred at {temperature} °C."

print(procedure)

A reaction vial was charged with 1-vinylnaphthalene (0.1 mmol, 0.015898 mL), morpholine (0.1 mmol, 0.008721 mL), potassium persulfate (0.2 mmol, 0.054066 g), sodium chloride (0.15 mmol, 0.008766 g). The mixture was dissolved in acetonitrile (1 mL) and stirred at 60 °C.
